# Zero-shot runs on a Colab GPU

Runs `arena run zero-shot` on a free Colab GPU and keeps the score parts on
Google Drive, so a disconnected session loses at most the day in progress.
Start it again and it skips the days already scored.

Before the first run:

1. Choose *Runtime → Change runtime type → T4 GPU*.
2. Zip the local cache folder `data/cache` as `cache.zip` (the archive holds
   the folder `cache/`) and upload it to `MyDrive/tsfm-weather-arena/`.

When it finishes, copy `MyDrive/tsfm-weather-arena/parts/` into
`results/parts/` of the local repository and run the same command there:
every day is already scored, so it only rebuilds the summary.

In [ ]:
BRANCH = "phase-4-zero-shot"
REGIME = "b"  # a (15 minutes) or b (6 hours)
SPLIT = "smoke"  # smoke (validation days) or test
RUNS = ""  # comma-separated run names from configs/tsfm.yaml; empty runs all
DRIVE = "/content/drive/MyDrive/tsfm-weather-arena"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone --quiet --depth 1 --branch {BRANCH} https://github.com/SashaRisoluti/tsfm-weather-arena.git /content/arena
%cd /content/arena
!pip install --quiet uv
!uv sync --quiet --locked --group tsfm
!uv sync --quiet --locked --project external/timestral
!uv run --no-sync python -c "import torch; assert torch.cuda.is_available(), 'no CUDA device'; print(torch.__version__, torch.cuda.get_device_name(0))"

In [ ]:
!mkdir -p data results {DRIVE}/parts
!unzip -q -o {DRIVE}/cache.zip -d data
!ln -sfn {DRIVE}/parts results/parts
!du -sh data/cache

In [ ]:
runs = f"--runs {RUNS}" if RUNS else ""
!uv run --no-sync arena run zero-shot --regime {REGIME} --split {SPLIT} {runs}
!mkdir -p {DRIVE}/summary && cp results/summary/*.csv {DRIVE}/summary/